# Applying `MCube` to the 10x Xenium CRC dataset

In [1]:
set.seed(20250502)

library(Matrix)
library(ggplot2)

library(spacexr)
library(MCube)

In [2]:
RAW_DATA_PATH <- "/import/home/share/zw/data/CRC"
DATA_PATH <- "/import/home/share/zw/pql/data/CRC"
RESULT_PATH <- "/import/home/share/zw/pql/results/CRC"

if (!dir.exists(file.path(RESULT_PATH, "Xenium"))) {
    dir.create(file.path(RESULT_PATH, "Xenium"), recursive = TRUE)
}

## Cell type deconvolution using `RCTD`

In [3]:
# library(Seurat)

# FlexRef <- Read10X_h5(file.path(
#     RAW_DATA_PATH, "sc", "HumanColonCancer_Flex_Multiplex_count_filtered_feature_bc_matrix.h5"
# ))
# # MetaData <- readRDS(file.path(
# #     RAW_DATA_PATH, "sc", "FlexSeuratV5_MetaData.rds"
# # )) # See FlexSingleCell.R if not generated.

# meta <- read.csv(file.path(
#     RAW_DATA_PATH, "HumanColonCancer_VisiumHD/MetaData/SingleCell_MetaData.csv.gz"
# ))

# KpIdents <- names(which(table(meta$Level2) > 25))
# meta <- meta[meta$Level2 %in% KpIdents, ]
# FlexRef <- FlexRef[, meta$Barcode]

# CTRef <- meta$Level2
# CTRef <- gsub("/", "_", CTRef)
# CTRef <- as.factor(CTRef)
# names(CTRef) <- meta$Barcode

# reference <- Reference(FlexRef, CTRef, colSums(FlexRef))

In [4]:
# counts <- as.data.frame(readr::read_csv(
#     file.path(DATA_PATH, "Xenium", "xenium_p2_counts.csv")
# ))
# rownames(counts) <- counts[, 1]
# counts[, 1] <- NULL
# # head(counts)

# coordinates <- as.data.frame(readr::read_csv(
#     file.path(DATA_PATH, "Xenium", "xenium_p2_coordinates.csv")
# ))
# rownames(coordinates) <- coordinates[, 1]
# coordinates[, 1] <- NULL
# head(coordinates)
# coordinates$x <- sum(range(coordinates$x)) - coordinates$x
# # head(coordinates)

# nUMI <- rowSums(counts)

# puck <- SpatialRNA(coordinates, t(counts), nUMI)

# myRCTD_xenium <- create.RCTD(puck, reference, max_cores = 8)
# myRCTD_xenium <- run.RCTD(myRCTD_xenium, doublet_mode = "doublet")

# saveRDS(
#     myRCTD_xenium,
#     file = file.path(
#         RESULT_PATH, "Xenium", "myRCTD.rds"
#     )
# )

## Cell-type-specific SVG identification using `MCube`

Due to the high resolution of the Xenium data, for the cell types of interest, we select bins that are confirmed to contain those specific cell types based on the results from `RCTD` (doublet mode) for further analysis.

In [5]:
myRCTD <- readRDS(file.path(RESULT_PATH, "Xenium", "myRCTD.rds"))
weights_RCTD <- as.matrix(myRCTD@results$weights)
proportions_RCTD <- weights_RCTD / rowSums(weights_RCTD)
spot_effects_RCTD <- log(rowSums(weights_RCTD))
names(spot_effects_RCTD) <- rownames(weights_RCTD)
doublet_results_RCTD <- myRCTD@results$results_df

In [6]:
sample_size_max <- 10000
celltype_threshold <- 100
for (celltype in colnames(proportions_RCTD)) {
    spots_used <- rownames(doublet_results_RCTD)[
        ((doublet_results_RCTD$spot_class == "singlet" |
            doublet_results_RCTD$spot_class == "doublet_uncertain") &
            doublet_results_RCTD$first_type == celltype
        ) |
            (doublet_results_RCTD$spot_class == "doublet_certain" &
                (doublet_results_RCTD$first_type == celltype |
                    doublet_results_RCTD$second_type == celltype))
    ]

    if (length(spots_used) > 0 & sum(proportions_RCTD[spots_used, celltype]) > celltype_threshold) {
        if (length(spots_used) > sample_size_max) {
            spots_used <- sample(spots_used, size = sample_size_max, replace = FALSE)
        }

        mcube_object <- createMCube(
            counts = t(as.matrix(myRCTD@originalSpatialRNA@counts[, spots_used])),
            coordinates = as.matrix(myRCTD@spatialRNA@coords[spots_used, ]),
            proportions = proportions_RCTD[spots_used, ],
            library_sizes = myRCTD@spatialRNA@nUMI[spots_used],
            reference = t(myRCTD@cell_type_info$info[[1]]),
            used_for_deconvolution = rownames(myRCTD@spatialRNA@counts),
            spot_effects = spot_effects_RCTD[spots_used],
            celltype_test = celltype,
            proportion_threshold = 0.01
        )
        mcube_object <- mcubeFitNull(
            mcube_object,
            num_workers = 35, num_threads = 2
        )
        mcube_object <- mcubeTest(
            mcube_object,
            num_workers = 35, num_threads = 2, shared_memory = TRUE
        )

        saveRDS(
            mcube_object,
            file = file.path(
                RESULT_PATH, "Xenium",
                paste0("mcube_", celltype, ".rds")
            )
        )
    }
}

The batch_id is not provided!
All spots are assumed to be from the same batch and share the same gene platform effects.

Select high-abundance cell types to analyze with proportion_threshold = 0.01 and celltype_threshold = 100.

mcubeFilterCellTypes: Cell type(s) CAF, CD4 T cell, CD8 Cytotoxic T cell, Endothelial, Enteric Glial, Fibroblast, Lymphatic Endothelial, Macrophage, Myofibroblast, Pericytes, Plasma, Proliferating Immune II, SM Stress Response, Smooth Muscle, Tumor III, Unknown III (SM), vSM pass the threshold.

Cell type(s) CAF will be analyzed.

Filter out lowly-expressed genes with gene_threshold = 5e-05.

mcubeFilterGenes: 367 genes pass the threshold.

The platform effects are not provided and need to be estimated from data!

Select highly-expressed genes to analyze for each specific cell type with reference_threshold = 0.5.

mcubeFilterGenesCellType: Select 52 genes to analyze for CAF.

Preprocessed data description: 10000 spots and 32 cell types in total. 10000 spots, 52